## 0. Install Dependencies

Run this cell first, then **restart the kernel**, then run all remaining cells.

In [ ]:
import subprocess, sys

packages = [
    "docling==2.96.0",
    "transformers==4.55.4",
    "sentence-transformers==5.1.0",
    "chromadb==1.0.20",
    "rank-bm25==0.2.2",
    "pymupdf==1.26.4",
    "groq==0.31.0",
    "openai==1.107.2",
    "tenacity==9.1.2",
    "tqdm==4.67.1",
    "pdfplumber==0.11.4",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-U"] + packages
)
print("Installation done.")


# Payer Policy PA Parameter Extraction Pipeline

End-to-end RAG pipeline -- extracts 12 Prior Authorization parameters per brand from PsO payer policy PDFs.

| Step | Description |
|------|-------------|
| 1 | Configuration and Imports |
| 2 | LLM Client (OpenRouter / Groq + in-memory cache) |
| 3 | PDF to Markdown (PyMuPDF + Docling) |
| 4 | Chunking and Vector Store (BM25 + BGE-768 + reranker) |
| 5 | Brand Detection |
| 6 | Parameter Extraction and Access Score |
| 7 | Run Pipeline |

> **Kaggle:** Add `OPENROUTER_API_KEY` and `LLM_PROVIDER` via *Add-ons > Secrets*, enable *Attach to session*.  
> **Local:** `export OPENROUTER_API_KEY=... LLM_PROVIDER=openrouter`

## 1. Configuration and Imports

In [ ]:
# torch must be imported and fully initialised BEFORE docling/transformers
import torch
import torch._utils
import torch.utils

import csv
import hashlib
import json
import logging
import os
import re
import shutil
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

from tqdm.auto import tqdm
import fitz
import chromadb
import numpy as np
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from openai import OpenAI
from groq import Groq
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tenacity import (
    retry, retry_if_exception_type, stop_after_attempt, wait_exponential,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
GROQ_API_KEY       = os.environ.get("GROQ_API_KEY", "")
LLM_PROVIDER       = os.environ.get("LLM_PROVIDER", "openrouter")

assert OPENROUTER_API_KEY or GROQ_API_KEY, "Set OPENROUTER_API_KEY or GROQ_API_KEY"
print(f"Provider : {LLM_PROVIDER}")
print(f"Key set  : {bool(OPENROUTER_API_KEY) if LLM_PROVIDER == 'openrouter' else bool(GROQ_API_KEY)}")

_HERE        = Path('.').resolve()
PDF_DIR      = _HERE / "pdfs" / "Sample_PsO_ADS_Track"
MARKDOWN_DIR = _HERE / "markdown_output"
OUTPUT_CSV   = _HERE / "extraction_output.csv"
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL     = "BAAI/bge-base-en-v1.5"   # 768-dim
RERANK_MODEL    = "BAAI/bge-reranker-v2-m3"
COLLECTION      = "payer_policy"
CHUNK_SIZE      = 900
CHUNK_OVERLAP   = 200
RRF_K           = 60
RETRIEVAL_TOP_K = 6
PARAM_TOP_K     = 4     # max chunks passed to 70B for param extraction
HEADER_RATIO    = 0.07
FOOTER_RATIO    = 0.07

PARAMS = [
    "Age",
    "Step Therapy Requirements Documented in Policy",
    "Number of Steps through Brands",
    "Number of Steps through Generic",
    "Step through-Phototherapy",
    "TB Test required",
    "Initial Authorization Duration(in-months)",
    "Reauthorization Duration(in-months)",
    "Reauthorization Required",
    "Reauthorization Requirements Documented in Policy",
    "Specialist Types",
    "Quantity Limits",
]
CSV_COLUMNS = ["filename", "brand"] + PARAMS + ["access_score"]

print(f"Chunk size : {CHUNK_SIZE}  overlap={CHUNK_OVERLAP}")
print(f"Embed model: {EMBED_MODEL}  (768-dim)")
print(f"PDF dir    : {PDF_DIR}")
print(f"Output CSV : {OUTPUT_CSV}")


## 2. LLM Client

OpenRouter and Groq both serve `llama-3.1-8b-instruct`.  
In-memory cache by MD5(messages+model) avoids duplicate API calls within a session.

In [ ]:
# ---------------------------------------------------------------------------
# Model registry — 8B for brand detection, 70B for param extraction
# ---------------------------------------------------------------------------
_MODELS = {
    "openrouter-8b":  "meta-llama/llama-3.1-8b-instruct",
    "openrouter-70b": "meta-llama/llama-3.1-70b-instruct",
    "groq-8b":        "llama-3.1-8b-instant",
    "groq-70b":       "llama-3.3-70b-versatile",
}

_llm_cache: Dict[str, str] = {}

def _key(messages: List[Dict], model: str) -> str:
    payload = json.dumps({"m": messages, "model": model}, sort_keys=True)
    return hashlib.md5(payload.encode()).hexdigest()


class LLMClient:
    """
    llm_8b  = LLMClient(provider="openrouter", size="8b")
    llm_70b = LLMClient(provider="openrouter", size="70b")
    """

    def __init__(self, provider: str = "openrouter", size: str = "8b"):
        provider = provider.lower()
        size     = size.lower()
        key      = f"{provider}-{size}"
        if key not in _MODELS:
            raise ValueError(f"Unknown provider/size: '{key}'. Valid: {list(_MODELS)}")

        self.provider = provider
        self.size     = size
        self.model    = _MODELS[key]

        if provider == "openrouter":
            self._client = OpenAI(
                api_key=OPENROUTER_API_KEY,
                base_url="https://openrouter.ai/api/v1",
            )
        else:
            self._client = Groq(api_key=GROQ_API_KEY)

        log.info("LLMClient — provider=%s  size=%s  model=%s", provider, size, self.model)

    def complete(
        self,
        messages: List[Dict],
        temperature: float = 0.0,
        max_tokens: int = 1024,
        use_llm_cache: bool = True,
    ) -> str:
        cache_key = _key(messages, self.model)
        if use_llm_cache and cache_key in _llm_cache:
            log.debug("Cache hit [%s]", cache_key[:8])
            return _llm_cache[cache_key]

        log.debug("Cache miss [%s] — calling API", cache_key[:8])
        response = self._call(messages, temperature, max_tokens)
        if use_llm_cache:
            _llm_cache[cache_key] = response
        return response

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=2, min=4, max=30),
        retry=retry_if_exception_type(Exception),
        reraise=True,
    )
    def _call(self, messages: List[Dict], temperature: float, max_tokens: int) -> str:
        resp = self._client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content.strip()

    @property
    def cache_size(self) -> int:
        return len(_llm_cache)

    def clear_cache(self) -> None:
        _llm_cache.clear()

print("LLMClient defined (8b + 70b per provider).")


## 3. PDF to Markdown

PyMuPDF cleans the PDF in memory (headers/footers/links/credentials), then Docling converts it to Markdown with table detection (forced CPU).

In [ ]:
HEADER_RATIO = 0.07
FOOTER_RATIO = 0.07

_CRED_PATTERNS = [
    re.compile(r"(username|user[\s_-]*name|login|user[\s_-]*id)\s*[:=]\s*\S+", re.I),
    re.compile(r"(password|passwd|pwd)\s*[:=]\s*\S+", re.I),
    re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"),
]

_REF_HEADING = re.compile(
    r"^\s*(?:references|bibliography|works cited)\s*$", re.I | re.M
)


# ---------------------------------------------------------------------------
# Cleaning helpers (operate on fitz.Document in place)
# ---------------------------------------------------------------------------
def _redact_zone(page: fitz.Page, zone: fitz.Rect) -> None:
    for b in page.get_text("blocks", clip=zone):
        page.add_redact_annot(fitz.Rect(b[:4]), fill=(1, 1, 1))


def _redact_credentials(page: fitz.Page) -> None:
    for block in page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE).get("blocks", []):
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                if any(p.search(span.get("text", "")) for p in _CRED_PATTERNS):
                    page.add_redact_annot(fitz.Rect(span["bbox"]), fill=(1, 1, 1))


def _strip_references(doc: fitz.Document) -> None:
    """Redact everything from the first References/Bibliography heading to end of doc."""
    cut_page: Optional[int] = None
    cut_y: float = 0.0

    for pno in range(doc.page_count):
        page = doc[pno]
        for b in page.get_text("blocks"):
            b_text = b[4].strip()
            if _REF_HEADING.match(b_text):
                cut_page = pno
                cut_y = float(b[1])
                break
        if cut_page is not None:
            break

    if cut_page is None:
        return

    # Redact from heading to bottom of that page
    page = doc[cut_page]
    w, h = page.rect.width, page.rect.height
    page.add_redact_annot(fitz.Rect(0, cut_y, w, h), fill=(1, 1, 1))
    page.apply_redactions()

    # Delete all subsequent pages
    total = doc.page_count
    if cut_page + 1 < total:
        doc.delete_pages(cut_page + 1, total - 1)

    log.info("[refs]    Stripped references section (from page %d)", cut_page + 1)


def _clean_doc(doc: fitz.Document) -> None:
    """Clean headers/footers/links/credentials in place."""
    for page in doc:
        h, w = page.rect.height, page.rect.width
        _redact_zone(page, fitz.Rect(0, 0, w, h * HEADER_RATIO))
        _redact_zone(page, fitz.Rect(0, h * (1 - FOOTER_RATIO), w, h))
        for link in page.get_links():
            page.delete_link(link)
        _redact_credentials(page)
        page.apply_redactions()


# ---------------------------------------------------------------------------
# Docling converter — always CPU so GPU stays free for embeddings/reranker
# ---------------------------------------------------------------------------
def build_converter() -> DocumentConverter:
    opts = PdfPipelineOptions()
    opts.do_ocr = False
    opts.do_table_structure = True
    opts.table_structure_options.do_cell_matching = True

    try:
        from docling.datamodel.pipeline_options import AcceleratorOptions
        from docling.datamodel.accelerator_options import AcceleratorDevice
        opts.accelerator_options = AcceleratorOptions(
            num_threads=4, device=AcceleratorDevice.CPU
        )
    except ImportError:
        pass

    return DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
    )


# ---------------------------------------------------------------------------
# Main function
# ---------------------------------------------------------------------------
def convert_pdf(
    pdf_path: Path,
    md_dir: Path,
    converter: DocumentConverter,
) -> Optional[Path]:
    """
    Clean *pdf_path* in memory, strip references, and convert to Markdown via Docling.
    Writes one .md file to *md_dir*.  Skips if markdown already exists.

    Returns the Path to the written markdown file, or None on failure.
    """
    md_out = md_dir / (pdf_path.stem + ".md")

    if md_out.exists():
        log.info("[skip]    %s — markdown already exists", pdf_path.name)
        return md_out

    tmp_path: Optional[Path] = None
    try:
        doc = fitz.open(str(pdf_path))
        _clean_doc(doc)
        _strip_references(doc)

        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
            tmp_path = Path(tmp.name)
            doc.save(str(tmp_path), garbage=4, deflate=True)
        doc.close()

        result   = converter.convert(str(tmp_path))
        tmp_path.unlink(missing_ok=True)

        document = result.document
        markdown = document.export_to_markdown()

        n_tables = 0
        try:
            from docling_core.types.doc import DocItemLabel
            n_tables = sum(
                1 for item, _ in document.iterate_items()
                if getattr(item, "label", None) == DocItemLabel.TABLE
            )
        except Exception:
            n_tables = markdown.count("\n|")

        md_dir.mkdir(parents=True, exist_ok=True)
        md_out.write_text(markdown, encoding="utf-8")
        log.info("[done]    %s  | tables: %d | chars: %d",
                 pdf_path.name, n_tables, len(markdown))
        return md_out

    except Exception as exc:
        log.error("[fail]    %s — %s", pdf_path.name, exc)
        if tmp_path and tmp_path.exists():
            tmp_path.unlink(missing_ok=True)
        return None

# ---------------------------------------------------------------------------
# pdfplumber — tabular PDF detection and extraction
# ---------------------------------------------------------------------------
def _is_tabular_pdf(pdf_path: Path, threshold: float = 0.5) -> bool:
    """Return True if page 1 is dominated by tables (>threshold of page area)."""
    try:
        import pdfplumber
        with pdfplumber.open(str(pdf_path)) as pdf:
            if not pdf.pages:
                return False
            page      = pdf.pages[0]
            page_area = page.width * page.height
            if page_area == 0:
                return False
            table_area = sum(
                (t.bbox[2] - t.bbox[0]) * (t.bbox[3] - t.bbox[1])
                for t in page.find_tables()
            )
            ratio = table_area / page_area
            log.info("[detect]  %s — table area ratio: %.2f", pdf_path.name, ratio)
            return ratio > threshold
    except Exception as e:
        log.warning("[detect]  %s — tabular check failed: %s", pdf_path.name, e)
        return False


def convert_pdf_tabular(pdf_path: Path, md_dir: Path) -> Optional[Path]:
    """
    Extract tabular PDFs (e.g. PA Detail sheets) using pdfplumber.
    Each drug row is converted to a labeled prose block so the LLM can read
    each field by name:

        ## ADALIMUMAB (HUMIRA)
        **Required Medical Information:** Diagnosis. For RA...
        **Age Restriction:** Member must be 2 years of age or older.
    """
    import pdfplumber

    md_out = md_dir / (pdf_path.stem + ".md")
    if md_out.exists():
        log.info("[skip]    %s — markdown already exists", pdf_path.name)
        return md_out

    def _clean(cell) -> str:
        if cell is None:
            return ""
        return " ".join(str(cell).split())

    _HEADER_MARKERS = {"Group", "Indication Indicator", "Required Medical Information"}

    try:
        headers: Optional[List[str]] = None
        rows:    List[List[str]]     = []

        with pdfplumber.open(str(pdf_path)) as pdf:
            for page in pdf.pages:
                for table in page.find_tables():
                    extracted = table.extract()
                    if not extracted:
                        continue
                    for row in extracted:
                        cleaned = [_clean(c) for c in row]
                        if headers is None and _HEADER_MARKERS & set(cleaned):
                            headers = cleaned
                            continue
                        if headers and cleaned == headers:
                            continue
                        if not any(cleaned):
                            continue
                        rows.append(cleaned)

        if not rows:
            log.warning("[tabular] %s — no rows extracted by pdfplumber", pdf_path.name)
            return None

        lines = ["# Prior Authorization Detail\n"]
        for row in rows:
            if headers:
                row = (row + [""] * len(headers))[:len(headers)]
            brand = row[0] if row else "Unknown"
            lines.append(f"\n## {brand}\n")
            col_range = enumerate(headers[1:], start=1) if headers else enumerate(row[1:], start=1)
            for i, header in col_range:
                val = row[i] if i < len(row) else ""
                if val:
                    lines.append(f"**{header}:** {val}")

        markdown = "\n".join(lines)
        md_dir.mkdir(parents=True, exist_ok=True)
        md_out.write_text(markdown, encoding="utf-8")
        log.info("[tabular] %s  | drugs: %d | chars: %d",
                 pdf_path.name, len(rows), len(markdown))
        return md_out

    except Exception as exc:
        log.error("[fail]    %s (pdfplumber) — %s", pdf_path.name, exc)
        return None


# Patch convert_pdf to route tabular PDFs to pdfplumber
_orig_convert_pdf = convert_pdf

def convert_pdf(pdf_path: Path, md_dir: Path, converter: DocumentConverter) -> Optional[Path]:
    """Route tabular PDFs to pdfplumber, document-style PDFs to Docling."""
    md_out = md_dir / (pdf_path.stem + ".md")
    if md_out.exists():
        log.info("[skip]    %s — markdown already exists", pdf_path.name)
        return md_out

    if _is_tabular_pdf(pdf_path):
        log.info("[route]   %s — tabular PDF → pdfplumber", pdf_path.name)
        result = convert_pdf_tabular(pdf_path, md_dir)
        if result:
            return result
        log.warning("[route]   %s — pdfplumber failed, falling back to Docling", pdf_path.name)

    return _orig_convert_pdf(pdf_path, md_dir, converter)


print("PDF to Markdown functions defined (pdfplumber + Docling).")


## 4. Chunking and Vector Store

Recursive character splitting (900 chars / 150 overlap), ChromaDB storage, BM25 + dense cosine + RRF fusion + cross-encoder reranker.

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------



# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------
@dataclass
class Chunk:
    chunk_id: str
    text: str
    columns: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)


# ---------------------------------------------------------------------------
# Recursive text splitter
# ---------------------------------------------------------------------------
_SEPARATORS = ["\n## ", "\n### ", "\n#### ", "\n\n", "\n", ". ", " ", ""]


def _merge_splits(splits: List[str], sep: str, size: int, overlap: int) -> List[str]:
    """
    Merge atomic splits into chunks <= size chars.
    Sliding-window: flush when the next split would exceed size,
    pop from the front until retained context <= overlap.
    """
    chunks: List[str] = []
    window: List[str] = []
    window_len = 0
    sep_len = len(sep)

    def flush() -> None:
        if not window:
            return
        chunk = sep.join(window)
        if len(chunk) <= size:
            chunks.append(chunk)
        else:
            step = max(1, size - overlap)
            for i in range(0, len(chunk), step):
                piece = chunk[i: i + size]
                if piece.strip():
                    chunks.append(piece)

    for s in splits:
        s_len = len(s)
        add_len = s_len + (sep_len if window else 0)

        if window_len + add_len > size:
            flush()
            while window and window_len > overlap:
                removed = window.pop(0)
                window_len -= len(removed) + (sep_len if window else 0)

        window.append(s)
        window_len += s_len + (sep_len if len(window) > 1 else 0)

    flush()
    return chunks


def _recursive_split(text: str, size: int = CHUNK_SIZE,
                     overlap: int = CHUNK_OVERLAP,
                     seps: List[str] = _SEPARATORS) -> List[str]:
    """
    Recursively split text using separators from coarsest to finest,
    then merge into chunks of at most `size` chars with `overlap`.
    """
    if not text.strip():
        return []
    if len(text) <= size:
        return [text.strip()]

    sep = None
    remaining_seps: List[str] = []
    for i, s in enumerate(seps):
        if s == "" or s in text:
            sep = s
            remaining_seps = seps[i + 1:]
            break

    if sep is None:
        return [text.strip()]

    if sep == "":
        step = max(1, size - overlap)
        return [text[i: i + size].strip()
                for i in range(0, len(text), step)
                if text[i: i + size].strip()]

    flat: List[str] = []
    for p in text.split(sep):
        p = p.strip()
        if not p:
            continue
        if len(p) <= size:
            flat.append(p)
        else:
            flat.extend(_recursive_split(p, size, overlap, remaining_seps))

    join_sep = sep.strip() or " "
    return [c.strip() for c in _merge_splits(flat, join_sep, size, overlap) if c.strip()]


# ---------------------------------------------------------------------------
# Table helpers
# ---------------------------------------------------------------------------
_TABLE_RE = re.compile(
    r"(\|[^\n]+\|\n\|[-| :]+\|\n(?:\|[^\n]+\|\n)*)",
    re.MULTILINE,
)
_HEADER_RE = re.compile(r"^#{1,4}\s+(.+)", re.MULTILINE)


def _extract_columns(table_text: str) -> List[str]:
    first_line = table_text.strip().splitlines()[0]
    return [c.strip() for c in first_line.split("|") if c.strip()]


def _split_table(table_text: str, columns: List[str],
                 size: int, meta: Dict) -> List[Chunk]:
    """Split an oversized table into row-batches, each prefixed with the header row."""
    lines = table_text.strip().splitlines()
    if len(lines) < 3:
        return []

    header_block = lines[0] + "\n" + lines[1] + "\n"
    chunks: List[Chunk] = []
    buf = header_block

    for row in lines[2:]:
        candidate = buf + row + "\n"
        if len(candidate) <= size:
            buf = candidate
        else:
            if buf.strip() != header_block.strip():
                chunks.append(Chunk(text=buf.strip(), columns=columns,
                                    metadata=meta, chunk_id=""))
            buf = header_block + row + "\n"

    if buf.strip() and buf.strip() != header_block.strip():
        chunks.append(Chunk(text=buf.strip(), columns=columns,
                            metadata=meta, chunk_id=""))
    return chunks


# ---------------------------------------------------------------------------
# Markdown chunker
# ---------------------------------------------------------------------------
def chunk_markdown(md_text: str, pdf_name: str) -> List[Chunk]:
    """Parse one markdown document into Chunk objects."""
    chunks: List[Chunk] = []
    current_header = "Introduction"
    idx = 0

    def _next_id() -> str:
        nonlocal idx
        cid = f"{Path(pdf_name).stem}_{idx:04d}"
        idx += 1
        return cid

    segments: List[Dict] = []
    last_end = 0
    for m in _TABLE_RE.finditer(md_text):
        if m.start() > last_end:
            segments.append({"type": "prose", "text": md_text[last_end:m.start()]})
        segments.append({"type": "table", "text": m.group(0)})
        last_end = m.end()
    if last_end < len(md_text):
        segments.append({"type": "prose", "text": md_text[last_end:]})

    for seg in segments:
        # ---- TABLE --------------------------------------------------------
        if seg["type"] == "table":
            columns = _extract_columns(seg["text"])
            meta = {"table": True, "pdf": pdf_name, "header": current_header}
            if len(seg["text"]) <= CHUNK_SIZE:
                chunks.append(Chunk(
                    chunk_id=_next_id(),
                    text=seg["text"].strip(),
                    columns=columns,
                    metadata=meta,
                ))
            else:
                for c in _split_table(seg["text"], columns, CHUNK_SIZE, meta):
                    c.chunk_id = _next_id()
                    chunks.append(c)

        # ---- PROSE --------------------------------------------------------
        else:
            prose = seg["text"]
            for hdr in _HEADER_RE.finditer(prose):
                current_header = hdr.group(1).strip()

            for split in _recursive_split(prose):
                for hdr in _HEADER_RE.finditer(split):
                    current_header = hdr.group(1).strip()
                chunks.append(Chunk(
                    chunk_id=_next_id(),
                    text=split,
                    columns=[],
                    metadata={"table": False, "pdf": pdf_name, "header": current_header},
                ))

    return chunks


# ---------------------------------------------------------------------------
# ChromaDB store + hybrid search
# ---------------------------------------------------------------------------
class PolicyStore:
    """
    Models (encoder + reranker) are loaded ONCE at construction and reused
    across all PDFs — avoids repeated GPU allocations that cause CUDA OOM.

    Call init_collection() before processing each new PDF to get a fresh
    in-memory ChromaDB collection without reloading the models.
    """

    def __init__(self, embed_model: str = EMBED_MODEL, rerank_model: str = RERANK_MODEL):
        log.info("Loading embedding model: %s  (loaded once, shared across all PDFs)", embed_model)
        self.encoder = SentenceTransformer(embed_model, device=DEVICE)
        log.info("Embedding dim: %d", self.encoder.get_sentence_embedding_dimension())

        log.info("Loading reranker: %s", rerank_model)
        self.reranker = CrossEncoder(rerank_model, device=DEVICE)

        self.col    = None
        self.client = None
        self._bm25: Optional[BM25Okapi] = None
        self._bm25_ids: List[str] = []
        self._bm25_texts: List[str] = []
        log.info("PolicyStore ready — call init_collection() before each PDF")

    def init_collection(self) -> None:
        """Reset to a fresh in-memory ChromaDB collection. No disk I/O, no model reload."""
        self.client = chromadb.EphemeralClient()
        self.col    = self.client.get_or_create_collection(
            name=COLLECTION,
            metadata={"hnsw:space": "cosine"},
        )
        self._bm25       = None
        self._bm25_ids   = []
        self._bm25_texts = []
        log.info("Collection reset — fresh ephemeral store ready")

    def add_chunks(self, chunks: List[Chunk], batch_size: int = 64) -> int:
        existing = set(self.col.get(include=[])["ids"])
        new = [c for c in chunks if c.chunk_id not in existing]
        if not new:
            return 0

        for i in range(0, len(new), batch_size):
            batch = new[i: i + batch_size]
            texts = [c.text for c in batch]
            embeddings = self.encoder.encode(
                texts, batch_size=32, show_progress_bar=False,
                normalize_embeddings=True,
            ).tolist()
            self.col.add(
                ids=[c.chunk_id for c in batch],
                documents=texts,
                embeddings=embeddings,
                metadatas=[
                    {
                        **c.metadata,
                        "columns": "|".join(c.columns),
                        "table": str(c.metadata.get("table", False)),
                    }
                    for c in batch
                ],
            )

        self._bm25 = None
        log.info("Stored %d new chunks (%d already present)", len(new), len(existing))
        return len(new)

    def _ensure_bm25(self) -> None:
        if self._bm25 is not None:
            return
        result = self.col.get(include=["documents"])
        self._bm25_ids = result["ids"]
        self._bm25_texts = result["documents"]
        self._bm25 = BM25Okapi([t.lower().split() for t in self._bm25_texts])
        log.info("BM25 index built over %d docs", len(self._bm25_ids))

    def hybrid_search(self, query: str, top_k: int = 10,
                      rerank_candidates: int = 30) -> List[Dict[str, Any]]:
        """
        Three-stage retrieval:
          1. BM25 + dense cosine → RRF → top `rerank_candidates`
          2. bge-reranker-v2-m3 cross-encoder → re-scores each candidate
          3. Return top `top_k` by reranker score
        """
        self._ensure_bm25()
        n_candidates = min(rerank_candidates, max(self.col.count(), 1))

        # --- Stage 1a: BM25 sparse ---
        bm25_scores = self._bm25.get_scores(query.lower().split())
        sparse_ranks = np.argsort(bm25_scores)[::-1][:n_candidates].tolist()

        # --- Stage 1b: dense cosine ---
        q_vec = self.encoder.encode([query], normalize_embeddings=True).tolist()
        dense = self.col.query(
            query_embeddings=q_vec,
            n_results=n_candidates,
            include=["documents", "metadatas", "distances"],
        )
        dense_ids: List[str] = dense["ids"][0]

        # --- Stage 1c: RRF fusion ---
        rrf: Dict[str, float] = {}
        for rank, arr_idx in enumerate(sparse_ranks):
            cid = self._bm25_ids[arr_idx]
            rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (RRF_K + rank + 1)
        for rank, cid in enumerate(dense_ids):
            rrf[cid] = rrf.get(cid, 0.0) + 1.0 / (RRF_K + rank + 1)

        candidate_ids = sorted(rrf, key=lambda x: rrf[x], reverse=True)[:n_candidates]

        # Fetch full text for all candidates
        fetched = self.col.get(ids=candidate_ids, include=["documents", "metadatas"])
        id_map = {
            cid: (doc, meta)
            for cid, doc, meta in zip(
                fetched["ids"], fetched["documents"], fetched["metadatas"]
            )
        }

        # --- Stage 2: rerank ---
        pairs = [(query, id_map[cid][0]) for cid in candidate_ids if cid in id_map]
        rerank_scores = self.reranker.predict(pairs)   # shape: (n_candidates,)

        ranked = sorted(
            zip(candidate_ids, rerank_scores),
            key=lambda x: x[1],
            reverse=True,
        )[:top_k]

        # --- Stage 3: build results ---
        return [
            {
                "chunk_id": cid,
                "text": id_map[cid][0],
                "columns": [c for c in id_map[cid][1].get("columns", "").split("|") if c],
                "metadata": {
                    "table":  id_map[cid][1].get("table") == "True",
                    "pdf":    id_map[cid][1].get("pdf", ""),
                    "header": id_map[cid][1].get("header", ""),
                },
                "rrf_score":    round(rrf.get(cid, 0.0), 6),
                "rerank_score": round(float(score), 4),
            }
            for cid, score in ranked if cid in id_map
        ]


# ---------------------------------------------------------------------------
# Entry point — processes ONE markdown file
# ---------------------------------------------------------------------------

print("Chunking and PolicyStore defined.")

## 5. Brand Detection

Reads the full markdown (head 67% + tail 33%, max 6000 chars) and asks the LLM to identify all brands relevant to PsO.

In [ ]:
# Max characters sent to the 8B LLM for brand detection
MAX_CHARS = 4000

_DRUG_LIST_QUERIES = [
    "applicable drug list biologics PsO formulary covered medications",
    "preferred non-preferred biologic brand names plaque psoriasis",
    "step therapy biologic agents psoriasis prior authorization criteria",
]

_BRAND_PROMPT = """You are an expert at extracting structured prior authorization policy information from payer policy documents.

Your task is to identify all brands/products in this policy document that are relevant to plaque psoriasis (PsO).

Instructions:
1. Read the provided policy text carefully.
2. Identify all products listed in the Applicable Drug List or equivalent drug list section.
3. Determine whether the policy contains coverage criteria for plaque psoriasis (PsO).
4. Return every product/brand that is relevant to PsO extraction.
5. Include preferred/non-preferred status if explicitly stated.
6. Do not infer brands that are not explicitly listed.
7. If the policy has multiple indications, only identify brands relevant to PsO.
8. Return the brand name in CAPITAL LETTERS only.
9. Return ONLY the brand name — strip any generic/chemical name in parentheses.
   Example: "Tremfya (guselkumab)" -> "TREMFYA"

Return strict JSON only in this format:
{{
  "policy_has_pso": "Yes | No",
  "brands_relevant_to_pso": [
    {{
      "brand": "",
      "preferred_status": "Preferred | Non-preferred | Unspecified"
    }}
  ]
}}

Policy Text:
{policy_text}"""


def _clean_brand_name(name: str) -> str:
    """Strip generic name in parentheses and uppercase the brand."""
    name = re.sub(r"\s*\(.*?\)", "", name).strip()
    return name.upper()


def _parse_json(raw: str, pdf_name: str) -> Dict[str, Any]:
    # Strip markdown code fences (LLM sometimes wraps JSON in ```json ... ```)
    cleaned = re.sub(r"```(?:json)?\s*", "", raw).strip()
    try:
        start  = cleaned.index("{")
        end    = cleaned.rindex("}") + 1
        result = json.loads(cleaned[start:end])
        for b in result.get("brands_relevant_to_pso", []):
            b["brand"] = _clean_brand_name(b.get("brand", ""))
        return result
    except (ValueError, json.JSONDecodeError):
        log.error("Brand JSON parse failed for %s:\n%s", pdf_name, raw[:300])
        return {"policy_has_pso": "No", "brands_relevant_to_pso": []}


def detect_brands(md_path: Path, store: "PolicyStore", llm: LLMClient) -> Dict[str, Any]:
    """
    Stage 1 — Hybrid-search for drug-list chunks → pass to 8B LLM → brand list.
    Stage 2 — Per-brand anchor search → store anchor_chunk_ids for downstream retrieval.
    """
    pdf_name = md_path.stem + ".pdf"

    # Stage 1: find drug-list sections via hybrid search
    seen: set = set()
    context_chunks: List[Dict] = []
    for query in _DRUG_LIST_QUERIES:
        for r in store.hybrid_search(query, top_k=3):
            if r["metadata"]["pdf"] == pdf_name and r["chunk_id"] not in seen:
                seen.add(r["chunk_id"])
                context_chunks.append(r)

    if context_chunks:
        policy_text = "\n\n---\n\n".join(r["text"] for r in context_chunks)
        if len(policy_text) > MAX_CHARS:
            policy_text = policy_text[:MAX_CHARS]
        log.info("Brand detection — %s  (%d chunks, %d chars → 8B)", pdf_name, len(context_chunks), len(policy_text))
    else:
        # Fallback: use head of markdown
        raw_md = md_path.read_text(encoding="utf-8")
        policy_text = raw_md[:MAX_CHARS]
        log.info("Brand detection — %s  (no chunks found, using markdown head)", pdf_name)

    messages = [{"role": "user", "content": _BRAND_PROMPT.format(policy_text=policy_text)}]
    raw      = llm.complete(messages, temperature=0.0, max_tokens=4096)
    result   = _parse_json(raw, pdf_name)

    log.info("  policy_has_pso=%s  brands=%d",
             result.get("policy_has_pso", "?"),
             len(result.get("brands_relevant_to_pso", [])))

    # Stage 2: per-brand anchor chunk search — collect IDs for tier-1 retrieval
    for brand in result.get("brands_relevant_to_pso", []):
        brand_name   = brand["brand"]
        anchor_query = (
            f"{brand_name} prior authorization criteria plaque psoriasis "
            "step therapy reauthorization approval"
        )
        anchor_ids: List[str] = []
        for r in store.hybrid_search(anchor_query, top_k=2):
            if r["metadata"]["pdf"] == pdf_name:
                anchor_ids.append(r["chunk_id"])
        brand["anchor_chunk_ids"] = anchor_ids
        log.info("  Brand %-20s  anchors=%d", brand_name, len(anchor_ids))

    return result

print("Brand detection defined.")


## 6. Parameter Extraction and Access Score

One LLM call per brand using brand-specific retrieved chunks.

**Access Score (1-100):** weighted sum -- higher = easier access (less restrictive).

| Parameter | Weight |
|-----------|--------|
| Steps through Brands | 20 |
| Initial Auth Duration | 15 |
| TB Test required | 15 |
| Age | 10 |
| Steps through Generic | 10 |
| Step through-Phototherapy | 5 |
| Step Therapy text | 5 |
| Reauth Required | 5 |
| Reauth Duration | 5 |
| Specialist Types | 4 |
| Reauth Requirements text | 3 |
| Quantity Limits | 3 |

In [ ]:
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Prompt — one brand at a time
# ---------------------------------------------------------------------------
_PARAM_PROMPT = """\
You are an expert in extracting structured prior authorization policy data from payer policy documents.

Extract 12 PsO-specific parameters for the brand below using ONLY the provided policy chunks.

BRAND:
  Name             : {brand_name}
  Preferred status : {preferred_status}

INSTRUCTIONS:
- Extract for plaque psoriasis (PsO) only. Ignore other indications.
- If moderate-to-severe and severe PsO are distinguished, use moderate-to-severe criteria only.
- Universal criteria that apply to all brands must be combined with brand-specific criteria using AND logic.
- If OR statements exist, choose the least restrictive valid path.
- Count only what is explicitly stated. Do not infer.
- Use "NA" for any value not mentioned, unless rules below specify otherwise.
- Output strict JSON only — no explanation.

PARAMETERS:

1. Age: Age threshold for eligibility. Output "FDA labelled age" if only FDA labelling is mentioned. If two age groups listed, capture the youngest.

2. Step Therapy Requirements Documented in Policy: Full free-text of all step therapy language relevant to PsO for this brand (universal + brand-specific).

3. Number of Steps through Brands: Count of branded/biologic steps required before this brand is approved. Choose least restrictive OR path. Exclude phototherapy. "NA" if none.

4. Number of Steps through Generic: Count of non-biologic/generic/topical steps required. Exclude phototherapy. "NA" if none.

5. Step through-Phototherapy: "Yes" if phototherapy is a mandatory step. "No" if not required. "N/A" if no criteria at all.

6. TB Test required: "Y" if required. "N" if explicitly not required. "NA" if not mentioned.

7. Initial Authorization Duration(in-months): Numeric months. "Unspecified" if required but not stated numerically.

8. Reauthorization Duration(in-months): Numeric months. "Unspecified" if required but not stated numerically.

9. Reauthorization Required: "Yes" if reauth is documented. "No" if explicitly not required. "NA" otherwise.

10. Reauthorization Requirements Documented in Policy: Actual continuation/renewal criteria text for PsO.

11. Specialist Types: Specialist type(s) acceptable for prescribing/managing PsO treatment.

12. Quantity Limits: Only explicitly stated quantity limits. Do NOT extract dosage language. "NA" if not stated.

OUTPUT FORMAT — strict JSON only:
{{
  "brand": "{brand_name}",
  "preferred_status": "{preferred_status}",
  "Age": "",
  "Step Therapy Requirements Documented in Policy": "",
  "Number of Steps through Brands": "",
  "Number of Steps through Generic": "",
  "Step through-Phototherapy": "",
  "TB Test required": "",
  "Initial Authorization Duration(in-months)": "",
  "Reauthorization Duration(in-months)": "",
  "Reauthorization Required": "",
  "Reauthorization Requirements Documented in Policy": "",
  "Specialist Types": "",
  "Quantity Limits": "",
  "evidence": {{
    "Age": "",
    "Step Therapy Requirements Documented in Policy": "",
    "Number of Steps through Brands": "",
    "Number of Steps through Generic": "",
    "Step through-Phototherapy": "",
    "TB Test required": "",
    "Initial Authorization Duration(in-months)": "",
    "Reauthorization Duration(in-months)": "",
    "Reauthorization Required": "",
    "Reauthorization Requirements Documented in Policy": "",
    "Specialist Types": "",
    "Quantity Limits": ""
  }}
}}

RELEVANT POLICY CHUNKS:
{chunks}"""

# ---------------------------------------------------------------------------
# Per-brand chunk retrieval queries
# ---------------------------------------------------------------------------
_COMMON_QUERIES = [
    "step therapy prior authorization criteria plaque psoriasis PsO",
    "initial authorization duration months reauthorization renewal continuation criteria",
    "TB test tuberculosis quantity limit specialist prescriber dermatologist",
]


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _get_brand_chunks(
    store: PolicyStore,
    pdf_name: str,
    brand_name: str,
    anchor_chunk_ids: Optional[List[str]] = None,
) -> str:
    """
    Two-tier retrieval — capped at PARAM_TOP_K chunks total:
      Tier 1: anchor chunk IDs collected during brand detection (guaranteed relevant).
      Tier 2: brand-specific hybrid search + common param queries to fill remaining slots.
    """
    seen:  set       = set()
    texts: List[str] = []

    # --- Tier 1: anchor chunks ---
    if anchor_chunk_ids:
        fetched  = store.col.get(ids=anchor_chunk_ids, include=["documents"])
        id_to_doc = dict(zip(fetched["ids"], fetched["documents"]))
        for aid in anchor_chunk_ids:
            if aid in id_to_doc and aid not in seen and len(texts) < PARAM_TOP_K:
                seen.add(aid)
                texts.append(id_to_doc[aid])
        log.info("    Tier-1 anchor chunks: %d", len(texts))

    # --- Tier 2: hybrid fill ---
    remaining = PARAM_TOP_K - len(texts)
    if remaining > 0:
        brand_query = (
            f"{brand_name} prior authorization criteria plaque psoriasis "
            "step therapy reauthorization approval"
        )
        for r in store.hybrid_search(brand_query, top_k=remaining + 4):
            if r["metadata"]["pdf"] == pdf_name and r["chunk_id"] not in seen:
                seen.add(r["chunk_id"])
                texts.append(r["text"])
                if len(texts) >= PARAM_TOP_K:
                    break

        for query in _COMMON_QUERIES:
            if len(texts) >= PARAM_TOP_K:
                break
            for r in store.hybrid_search(query, top_k=RETRIEVAL_TOP_K):
                if r["metadata"]["pdf"] == pdf_name and r["chunk_id"] not in seen:
                    seen.add(r["chunk_id"])
                    texts.append(r["text"])
                    if len(texts) >= PARAM_TOP_K:
                        break

        log.info("    Tier-2 hybrid fill: %d total chunks", len(texts))

    return "\n\n---\n\n".join(texts[:PARAM_TOP_K])


def _parse_brand_json(raw: str, brand_name: str) -> Dict[str, Any]:
    try:
        start = raw.index("{")
        end   = raw.rindex("}") + 1
        return json.loads(raw[start:end])
    except (ValueError, json.JSONDecodeError):
        log.error("JSON parse failed for brand '%s':\n%s", brand_name, raw[:200])
        return {}


def _score_age(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", "", "fda labelled age", "fda labeled age"):
        return 1.0
    m = re.search(r"(\d+)", v)
    if not m:
        return 0.7
    age = int(m.group(1))
    if age <= 6:   return 1.0
    if age <= 12:  return 0.9
    if age <= 18:  return 0.7
    return 0.4


def _score_steps(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", "", "0"):  return 1.0
    m = re.search(r"(\d+)", v)
    if not m:                  return 1.0
    n = int(m.group(1))
    return max(0.0, 1.0 - n * 0.3)


def _score_duration(val: str) -> float:
    v = val.strip().lower()
    if v in ("na", ""):        return 0.5
    if v == "unspecified":     return 0.5
    m = re.search(r"(\d+)", v)
    if not m:                  return 0.5
    months = int(m.group(1))
    if months >= 12:  return 1.0
    if months >= 6:   return 0.7
    if months >= 3:   return 0.4
    return 0.2


def _score_yesno(val: str, yes_score: float = 0.3, no_score: float = 1.0) -> float:
    v = val.strip().lower()
    if v in ("yes", "y"):  return yes_score
    if v in ("no", "n"):   return no_score
    return 0.7  # NA / unspecified


def _score_text_present(val: str) -> float:
    """No text / NA = easier access (1.0); text present = more restrictive (0.3)."""
    v = val.strip().lower()
    return 0.3 if v and v != "na" else 1.0


_WEIGHTS = {
    "Number of Steps through Brands":                20,
    "Initial Authorization Duration(in-months)":     15,
    "TB Test required":                              15,
    "Age":                                           10,
    "Number of Steps through Generic":               10,
    "Step through-Phototherapy":                      5,
    "Step Therapy Requirements Documented in Policy":  5,
    "Reauthorization Required":                       5,
    "Reauthorization Duration(in-months)":            5,
    "Specialist Types":                               4,
    "Reauthorization Requirements Documented in Policy": 3,
    "Quantity Limits":                                3,
}


def compute_access_score(row: Dict[str, str]) -> int:
    scorers = {
        "Age":                                            lambda v: _score_age(v),
        "Step Therapy Requirements Documented in Policy": lambda v: _score_text_present(v),
        "Number of Steps through Brands":                 lambda v: _score_steps(v),
        "Number of Steps through Generic":                lambda v: _score_steps(v),
        "Step through-Phototherapy":                      lambda v: _score_yesno(v),
        "TB Test required":                               lambda v: _score_yesno(v, yes_score=0.3, no_score=1.0),
        "Initial Authorization Duration(in-months)":      lambda v: _score_duration(v),
        "Reauthorization Duration(in-months)":            lambda v: _score_duration(v),
        "Reauthorization Required":                       lambda v: _score_yesno(v),
        "Reauthorization Requirements Documented in Policy": lambda v: _score_text_present(v),
        "Specialist Types":                               lambda v: _score_text_present(v),
        "Quantity Limits":                                lambda v: _score_text_present(v),
    }
    total = sum(
        _WEIGHTS[p] * scorers[p](row.get(p, "NA"))
        for p in _WEIGHTS
    )
    return round(total)


def _flatten_row(filename: str, brand_result: Dict) -> Dict[str, str]:
    row = {
        "filename": filename,
        "brand":    brand_result.get("brand", ""),
    }
    for p in PARAMS:
        row[p] = str(brand_result.get(p, "NA"))
    row["access_score"] = str(compute_access_score(row))
    return row


# ---------------------------------------------------------------------------
# Core extraction — one LLM call per brand (used by main.py)
# ---------------------------------------------------------------------------
def extract_params(
    md_path: Path,
    brand_data: Dict[str, Any],
    store: PolicyStore,
    llm: LLMClient,
) -> List[Dict[str, str]]:
    """
    For each brand: retrieve brand-specific chunks → LLM → 12 params.
    One focused LLM call per brand instead of one big call for all brands.
    """
    pdf_name = md_path.stem + ".pdf"
    brands   = brand_data.get("brands_relevant_to_pso", [])

    if not brands:
        log.info("  No brands — skipping")
        return []

    rows: List[Dict[str, str]] = []

    for brand in tqdm(brands, desc=f'  {pdf_name}', unit='brand'):
        brand_name  = brand["brand"]
        preferred   = brand.get("preferred_status", "Unspecified")
        anchor_ids  = brand.get("anchor_chunk_ids", [])
        log.info("  Extracting — %s (%s)", brand_name, preferred)

        # Retrieve chunks specific to this brand
        chunks = _get_brand_chunks(store, pdf_name, brand_name, anchor_ids)
        if not chunks.strip():
            log.warning("    No chunks retrieved for %s — skipping", brand_name)
            continue

        log.info("    Chunks: %d chars → LLM", len(chunks))

        messages = [{
            "role": "user",
            "content": _PARAM_PROMPT.format(
                brand_name=brand_name,
                preferred_status=preferred,
                chunks=chunks,
            ),
        }]

        raw    = llm.complete(messages, temperature=0.0, max_tokens=1024)
        result = _parse_brand_json(raw, brand_name)

        if result:
            rows.append(_flatten_row(pdf_name, result))

    log.info("  Total rows extracted: %d", len(rows))
    return rows


# ---------------------------------------------------------------------------
# Convenience wrapper — builds its own ephemeral store internally
# ---------------------------------------------------------------------------
def process_markdown(md_path: Path, llm: LLMClient) -> List[Dict[str, str]]:
    """Standalone entry: brand detection + store + extraction in one call."""
    pdf_name   = md_path.stem + ".pdf"
    brand_data = detect_brands(md_path, llm)

    if brand_data.get("policy_has_pso") != "Yes":
        log.info("  No PsO policy — skipping")
        return []

    tmp_dir = tempfile.mkdtemp(prefix="chroma_param_")
    try:
        store  = PolicyStore(chroma_dir=Path(tmp_dir))
        chunks = chunk_markdown(md_path.read_text(encoding="utf-8"), pdf_name)
        store.add_chunks(chunks)
        return extract_params(md_path, brand_data, store, llm)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


# ---------------------------------------------------------------------------
# CSV writer
# ---------------------------------------------------------------------------
def write_csv(rows: List[Dict], output_path: Path, append: bool = True) -> None:
    existing: set = set()
    if append and output_path.exists():
        with open(output_path, newline="", encoding="utf-8") as f:
            for r in csv.DictReader(f):
                existing.add((r.get("filename", ""), r.get("brand", "")))

    new_rows = [r for r in rows if (r["filename"], r["brand"]) not in existing]
    skipped  = len(rows) - len(new_rows)
    if skipped:
        log.info("  Skipped %d duplicate row(s) already in CSV", skipped)
    if not new_rows:
        log.info("  No new rows to write")
        return

    mode         = "a" if (append and output_path.exists()) else "w"
    write_header = not (append and output_path.exists())
    with open(output_path, mode, newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
        if write_header:
            writer.writeheader()
        writer.writerows(new_rows)
    log.info("CSV updated — %d new row(s) → %s", len(new_rows), output_path)


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

print("Parameter extraction and access score defined.")

## 7. Run Pipeline

Full end-to-end pipeline for a single PDF.

In [ ]:
def run_pipeline(
    pdf_path: Path,
    md_path:  Optional[Path],
    store:    "PolicyStore",
) -> None:
    log.info("=" * 65)
    log.info("PDF  : %s", pdf_path.name)
    log.info("=" * 65)

    # ------------------------------------------------------------------
    # STEP 1 — Markdown (pre-converted, passed in)
    # ------------------------------------------------------------------
    log.info("STEP 1 — Markdown: %s", md_path.name if md_path else "MISSING")
    if md_path is None:
        log.error("No markdown — aborting")
        return

    # ------------------------------------------------------------------
    # STEP 2 — Chunk & Store (fresh in-memory ChromaDB, models reused)
    # ------------------------------------------------------------------
    log.info("STEP 2 — Chunk & Store")
    pdf_name = pdf_path.stem + ".pdf"
    md_text  = md_path.read_text(encoding="utf-8")
    chunks   = chunk_markdown(md_text, pdf_name)
    log.info("  Chunks: %d", len(chunks))

    store.init_collection()     # fresh collection, same encoder/reranker on GPU
    store.add_chunks(chunks)

    # ------------------------------------------------------------------
    # STEP 3 — Brand Detection [8B]
    # ------------------------------------------------------------------
    log.info("STEP 3 — Brand Detection [8B]")
    provider = os.getenv("LLM_PROVIDER", "openrouter")
    llm_8b   = LLMClient(provider=provider, size="8b")
    brand_data = detect_brands(md_path, store, llm_8b)

    has_pso = brand_data.get("policy_has_pso", "No")
    brands  = brand_data.get("brands_relevant_to_pso", [])
    log.info("  policy_has_pso : %s", has_pso)
    log.info("  brands found   : %s", [b["brand"] for b in brands])

    if has_pso != "Yes" or not brands:
        log.info("  No PsO brands — nothing to extract")
        return

    # ------------------------------------------------------------------
    # STEP 4 — Parameter Extraction [70B]
    # ------------------------------------------------------------------
    log.info("STEP 4 — Parameter Extraction [70B]")
    llm_70b = LLMClient(provider=provider, size="70b")
    rows    = extract_params(md_path, brand_data, store, llm_70b)

    # ------------------------------------------------------------------
    # STEP 5 — Summary
    # ------------------------------------------------------------------
    if rows:
        log.info("STEP 5 — %d brand(s) extracted for %s", len(rows), pdf_path.name)
        for r in rows:
            log.info("    %-25s  steps_brand=%-4s  reauth=%-4s  TB=%s  score=%s",
                     r["brand"], r["Number of Steps through Brands"],
                     r["Reauthorization Required"], r["TB Test required"],
                     r["access_score"])
    else:
        log.info("  No rows extracted for %s", pdf_path.name)

    log.info("LLM cache: %d entries", llm_8b.cache_size)

print("run_pipeline defined.")


## 8. Execute

Processes every PDF in `PDF_DIR` sequentially. Already-converted markdowns are skipped (Step 1 caches them). Already-extracted `(filename, brand)` pairs are skipped in the CSV.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_PDFS        = None  # None = all; set an int to limit e.g. MAX_PDFS = 5
CONVERT_WORKERS = 4     # parallel Docling workers (CPU)

pdfs = sorted(PDF_DIR.glob('*.pdf'))
if not pdfs:
    raise FileNotFoundError(f'No PDFs found in {PDF_DIR}')

if MAX_PDFS is not None:
    pdfs = pdfs[:MAX_PDFS]

print(f'Found {len(pdfs)} PDF(s) to process')

# ── Phase 1: Convert all PDFs → Markdown in parallel (Docling on CPU) ────────
print(f'\nPhase 1: PDF → Markdown  (parallel, {CONVERT_WORKERS} workers)')

def _convert_worker(pdf: Path):
    conv = build_converter()           # one converter per thread (not thread-safe to share)
    md   = convert_pdf(pdf, MARKDOWN_DIR, conv)
    return pdf, md

md_map: dict = {}
with ThreadPoolExecutor(max_workers=CONVERT_WORKERS) as pool:
    futures = {pool.submit(_convert_worker, pdf): pdf for pdf in pdfs}
    with tqdm(total=len(pdfs), desc='Converting', unit='pdf') as pbar:
        for fut in as_completed(futures):
            pdf, md = fut.result()
            md_map[pdf] = md
            pbar.update(1)

converted = sum(1 for v in md_map.values() if v is not None)
print(f'Converted {converted}/{len(pdfs)} PDFs')

# ── Phase 2: Load models ONCE, reuse across all PDFs ─────────────────────────
# Fixes CUDA OOM: encoder + reranker stay on GPU, only the ChromaDB
# collection is swapped per PDF via store.init_collection().
print('\nPhase 2: Loading embedding models (once)...')
store = PolicyStore()                  # loads SentenceTransformer + CrossEncoder on GPU

print('\nPhase 2: Brand detection + parameter extraction')
for i, pdf in enumerate(pdfs):
    md_path = md_map.get(pdf)
    if md_path is None:
        log.error('Skipping %s — markdown conversion failed', pdf.name)
        continue
    print(f'\n[{i+1}/{len(pdfs)}] {pdf.name}')
    try:
        run_pipeline(pdf, md_path=md_path, store=store)
    except Exception as e:
        log.error('Pipeline failed for %s: %s', pdf.name, e)
        continue

print(f'\nAll done. Output: {OUTPUT_CSV}')
